# Quint specifications (string_therapy)

The **Quint source** for the UI session model lives in the next code cell as `STRING_THERAPY_UI_QNT`. The following cell writes it to `string_therapy/quint_specs/string_therapy_ui.qnt` (Quint’s CLI reads files). From a terminal you can sync the same way as CI: `pixi run quint-export`.

**Environment:** Start Lab from the Pixi env so `node`, `java`, and `npx` match `pixi.toml`:

```bash
pixi run jupyter-lab-solo   # or: pixi run jupyter-lab
```

**First time in this clone:** install the Quint CLI into `./node_modules` (same as `pixi run quint-install`):

```bash
pixi run quint-install
```

Then run the cells below (edit the spec string, run helpers so the `.qnt` file updates), or use `pixi run quint-typecheck` / `quint-run` / `quint-verify` from a terminal.

In [ ]:
# Quint module (canonical). Synced to ``string_therapy/quint_specs/string_therapy_ui.qnt`` by the next cell and by ``pixi run quint-export``.
STRING_THERAPY_UI_QNT = r"""
/**
 * Canonical source: ``appspecifications.ipynb`` (``STRING_THERAPY_UI_QNT``).
 * Abstract session model for the string_therapy web UI: dock icons **1** and **4**,
 * optional ``notebook_known`` after a successful chat → notebook injection (mirrors
 * ``SESSION_NOTEBOOK_KEY`` being meaningful).
 *
 * CLI: ``quint verify string_therapy/quint_specs/string_therapy_ui.qnt --invariants activeIconImplementedInv``
 */
module string_therapy_ui {

  pure val ICON1: int = 1
  pure val ICON4: int = 4
  pure val DOCK: Set[int] = Set(ICON1, ICON4)

  /// Expanded visualization (dock selection).
  var active: int
  /// Session has pinned notebook context for chat injection.
  var notebook_known: bool

  val state = { active: active, notebook_known: notebook_known }

  action init: bool = all {
    active' = ICON4,
    notebook_known' = false,
  }

  action open_lab: bool = all {
    active' = ICON4,
    notebook_known' = notebook_known,
  }

  action open_icon1: bool = all {
    active' = ICON1,
    notebook_known' = notebook_known,
  }

  action remember_notebook: bool = all {
    active' = active,
    notebook_known' = true,
  }

  action step: bool = any {
    open_lab,
    open_icon1,
    remember_notebook,
  }

  val activeIconImplementedInv: bool = active.in(DOCK)
}
"""

In [1]:
from __future__ import annotations

import os
import shutil
import subprocess
from pathlib import Path


def repo_root() -> Path:
    """Directory containing ``pixi.toml`` (walk up from the notebook cwd)."""
    p = Path.cwd().resolve()
    for _ in range(12):
        if (p / "pixi.toml").is_file():
            return p
        if p.parent == p:
            break
        p = p.parent
    raise RuntimeError(
        "pixi.toml not found: start Jupyter from the stringTherapy repo, "
        "or ``cd`` there before running cells."
    )


ROOT = repo_root()
SPEC = ROOT / "string_therapy" / "quint_specs" / "string_therapy_ui.qnt"


def write_quint_spec_file() -> Path:
    """Write ``STRING_THERAPY_UI_QNT`` to ``SPEC`` so Quint CLI has a ``.qnt`` file."""
    text = STRING_THERAPY_UI_QNT.strip() + "\n"
    SPEC.parent.mkdir(parents=True, exist_ok=True)
    SPEC.write_text(text, encoding="utf-8")
    return SPEC


write_quint_spec_file()


def quint_argv() -> list[str]:
    """Prefer ``./node_modules/.bin/quint``, then ``quint`` on PATH."""
    local = ROOT / "node_modules" / ".bin" / "quint"
    if local.is_file():
        return [str(local)]
    q = shutil.which("quint")
    if q:
        return [q]
    raise FileNotFoundError(
        "Quint CLI missing. In a terminal at the repo root run: pixi run quint-install"
    )


def quint_run(
    *args: str, check: bool = True, capture_output: bool = False
) -> subprocess.CompletedProcess[str]:
    cmd = quint_argv() + list(args)
    print("$", " ".join(cmd))
    return subprocess.run(
        cmd,
        cwd=ROOT,
        env=os.environ.copy(),
        text=True,
        check=check,
        capture_output=capture_output,
    )


ROOT, SPEC

(PosixPath('/root/python_projects/stringTherapy'),
 PosixPath('/root/python_projects/stringTherapy/string_therapy/quint_specs/string_therapy_ui.qnt'))

In [2]:
# Optional: install npm deps from this notebook (same as ``pixi run quint-install``)
if not (ROOT / "node_modules" / ".bin" / "quint").is_file():
    subprocess.run(["npm", "install"], cwd=ROOT, check=True)
else:
    print("Quint already present:", ROOT / "node_modules" / ".bin" / "quint")

Quint already present: /root/python_projects/stringTherapy/node_modules/.bin/quint


In [3]:
quint_run("typecheck", str(SPEC))

$ /root/python_projects/stringTherapy/node_modules/.bin/quint typecheck /root/python_projects/stringTherapy/string_therapy/quint_specs/string_therapy_ui.qnt


CompletedProcess(args=['/root/python_projects/stringTherapy/node_modules/.bin/quint', 'typecheck', '/root/python_projects/stringTherapy/string_therapy/quint_specs/string_therapy_ui.qnt'], returncode=0)

In [4]:
quint_run(
    "run",
    str(SPEC),
    "--invariant",
    "activeIconImplementedInv",
    "--max-samples",
    "500",
    "--max-steps",
    "30",
)

$ /root/python_projects/stringTherapy/node_modules/.bin/quint run /root/python_projects/stringTherapy/string_therapy/quint_specs/string_therapy_ui.qnt --invariant activeIconImplementedInv --max-samples 500 --max-steps 30
An example execution:

[State 0] { active: 4, notebook_known: false }

[State 1] { active: 1, notebook_known: false }

[State 2] { active: 1, notebook_known: false }

[State 3] { active: 4, notebook_known: false }

[State 4] { active: 1, notebook_known: false }

[State 5] { active: 4, notebook_known: false }

[State 6] { active: 1, notebook_known: false }

[State 7] { active: 4, notebook_known: false }

[State 8] { active: 4, notebook_known: true }

[State 9] { active: 4, notebook_known: true }

[State 10] { active: 1, notebook_known: true }

[State 11] { active: 1, notebook_known: true }

[State 12] { active: 1, notebook_known: true }

[State 13] { active: 1, notebook_known: true }

[State 14] { active: 1, notebook_known: true }

[State 15] { active: 1, notebook_known

CompletedProcess(args=['/root/python_projects/stringTherapy/node_modules/.bin/quint', 'run', '/root/python_projects/stringTherapy/string_therapy/quint_specs/string_therapy_ui.qnt', '--invariant', 'activeIconImplementedInv', '--max-samples', '500', '--max-steps', '30'], returncode=0)

In [5]:
# Symbolic verification (Apalache). Requires Java — provided by Pixi ``openjdk`` when Lab is started via ``pixi run``.
quint_run(
    "verify",
    str(SPEC),
    "--invariants",
    "activeIconImplementedInv",
    "--max-steps",
    "15",
)

$ /root/python_projects/stringTherapy/node_modules/.bin/quint verify /root/python_projects/stringTherapy/string_therapy/quint_specs/string_therapy_ui.qnt --invariants activeIconImplementedInv --max-steps 15
# Usage statistics is OFF. We care about your privacy.
# If you want to help our project, consider enabling statistics with config --enable-stats=true.

Output directory: /root/python_projects/stringTherapy/_apalache-out/server/2026-05-02T04-29-11_16381085303916201777
# APALACHE version: 0.56.1 | build: 70cdaf4                       I@04:29:11.920
Starting checker server on port 8822...                           I@04:29:11.943
The Apalache server is running on port 8822. Press Ctrl-C to stop.


May 02, 2026 4:29:13 AM com.google.protobuf.GeneratedMessage warnPre22Gencode
As of 2022/09/29 (release 21.7) makeExtensionsImmutable should not be called from protobuf gencode. If you are seeing this message, your gencode is vulnerable to a denial of service attack. You should regenerate your code using protobuf 25.6 or later. Use the latest version that meets your needs. However, if you understand the risks and wish to continue with vulnerable gencode, you can set the system property `-Dcom.google.protobuf.use_unsafe_pre22_gencode` on the command line to silence this warning. You also can set `-Dcom.google.protobuf.error_on_unsafe_pre22_gencode` to throw an error instead. See security vulnerability: https://github.com/protocolbuffers/protobuf/security/advisories/GHSA-h4h5-3hr4-j3g2


PASS #0: SanyParser                                               I@04:29:14.731
PASS #1: TypeCheckerSnowcat                                       I@04:29:15.331
 > Running Snowcat .::.                                           I@04:29:15.331
 > Your types are purrfect!                                       I@04:29:15.514
 > All expressions are typed                                      I@04:29:15.515
PASS #2: ConfigurationPass                                        I@04:29:15.516
  > Set the initialization predicate to q::init                   I@04:29:15.521
  > Set the transition predicate to q::step                       I@04:29:15.522
  > Set an invariant to q::inv                                    I@04:29:15.523
PASS #3: DesugarerPass                                            I@04:29:15.529
  > Desugaring...                                                 I@04:29:15.530
PASS #4: InlinePass                                               I@04:29:15.538
Leaving only relevant operat

State 0: Checking 1 state invariants                              I@04:29:16.353
State 0: state invariant 0 holds.                                 I@04:29:16.368
Step 0: picking a transition out of 1 transition(s)               I@04:29:16.374
State 1: Checking 1 state invariants                              I@04:29:16.389
State 1: state invariant 0 holds.                                 I@04:29:16.392
Step 1: picking a transition out of 3 transition(s)               I@04:29:16.395
State 2: Checking 1 state invariants                              I@04:29:16.415
State 2: state invariant 0 holds.                                 I@04:29:16.419
State 2: Checking 1 state invariants                              I@04:29:16.422
State 2: state invariant 0 holds.                                 I@04:29:16.425
Step 2: picking a transition out of 3 transition(s)               I@04:29:16.430
State 3: Checking 1 state invariants                              I@04:29:16.440
State 3: state invariant 0 h

CompletedProcess(args=['/root/python_projects/stringTherapy/node_modules/.bin/quint', 'verify', '/root/python_projects/stringTherapy/string_therapy/quint_specs/string_therapy_ui.qnt', '--invariants', 'activeIconImplementedInv', '--max-steps', '15'], returncode=0)